# 35. Profile quarantine and impute student ages

**Hard**

Return a dictionary of three DataFrames: null_counts (one row counting nulls in each original students column; empty input gives zeros), quarantined (original student rows whose age is null), and adults. To form adults, include students with a matching city record by unique id, fill null age using the mean of non-null ages from the original students input rounded to nearest integer with .5 rounded up, then retain age >=18. If every age is null, leave nulls unfilled. adults columns: id,name,age,city,code. Quarantined rows remain eligible for imputation.


In [0]:
%run ./_setup $exercise_id="35"


**Your solution:** add imports and return your result from `solve`.


In [0]:
from pyspark.sql import functions as F

def solve(students, cities):
    # Write your solution here.
    null_content= (
        students
        .withColumn("null_flag_id", F.when(F.col("id").isNull(), 1).otherwise(0))
        .withColumn("null_flag_name", F.when(F.col("name").isNull(), 1).otherwise(0))
        .withColumn("null_flag_age", F.when(F.col("age").isNull(), 1).otherwise(0))
        .agg(
            F.coalesce(F.sum("null_flag_id"),F.lit(0)).alias("id"),
            F.coalesce(F.sum("null_flag_name"),F.lit(0)).alias("name"),
            F.coalesce(F.sum("null_flag_age"),F.lit(0)).alias("age")
        )
    )

    quarantined= (
        students
        .filter(
            (F.col("age").isNull())
        )
    )

    null_age=(
        students
        .filter(F.col("age").isNotNull())
        .agg(F.round(F.mean("age"),0).alias("mean_age"))
    )

    adults= (
        students
        .join(cities,"id","left")
        .crossJoin(null_age)
        .withColumn("age",F.coalesce("age","mean_age").cast("long"))
        .drop("mean_age")
        .filter(F.col("age")>=18)                    
    )

    return {
        "null_counts":null_content,
        "quarantined":quarantined,
        "adults":adults
    }



In [0]:
from pyspark.sql import functions as F

def solve(students, cities):
    # Write your solution here.
    null_content= (
        students
        .agg(
            F.count(F.when(F.col("id").isNull(),1)).alias("id"),
            F.count(F.when(F.col("name").isNull(),1)).alias("name"),
            F.count(F.when(F.col("age").isNull(),1)).alias("age")
        )
    )

    quarantined= (
        students
        .filter(
            (F.col("age").isNull())
        )
    )

    null_age=(
        students
        .filter(F.col("age").isNotNull())
        .agg(F.round(F.mean("age"),0).alias("mean_age"))
    )

    adults= (
        students
        .join(cities,"id","left")
        .crossJoin(null_age)
        .withColumn("age",F.coalesce("age","mean_age").cast("long"))
        .drop("mean_age")
        .filter(F.col("age")>=18)                    
    )

    return {
        "null_counts":null_content,
        "quarantined":quarantined,
        "adults":adults
    }



**Check:** run below to test all cases and save progress.


In [0]:
check_report = check_solution(exercise_id, solve, spark)


In [0]:
show_progress(exercise_id, spark)
